## ESC-50 のデータ取得

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!git clone --depth 1 https://github.com/karolpiczak/ESC-50.git /content/ESC-50

Cloning into '/content/ESC-50'...
remote: Enumerating objects: 2018, done.
remote: Counting objects: 100% (2018/2018), done.
remote: Compressing objects: 100% (1798/1798), done.
remote: Total 2018 (delta 215), reused 2009 (delta 214), pack-reused 0 (from 0)
Receiving objects: 100% (2018/2018), 615.62 MiB | 29.50 MiB/s, done.
Resolving deltas: 100% (215/215), done.
Updating files: 100% (2011/2011), done.


In [3]:
!cd /content/ESC-50 && git rev-parse HEAD

33c8ce9eb2cf0b1c2f8bcf322eb349b6be34dbb6


In [4]:
!mkdir -p "/content/drive/MyDrive/TRON/AI/dataset/raw/esc50"
!cp -r /content/ESC-50/audio "/content/drive/MyDrive/TRON/AI/dataset/raw/esc50"
!cp -r /content/ESC-50/meta "/content/drive/MyDrive/TRON/AI/dataset/raw/esc50"
!cp /content/ESC-50/LICENSE "/content/drive/MyDrive/TRON/AI/dataset/raw/esc50"

In [6]:
from pathlib import Path
import pandas as pd

root = Path("/content/drive/MyDrive/TRON/AI/dataset/raw/esc50")

audio_files = list((root / "audio").glob("*.wav"))
meta = pd.read_csv(root / "meta" / "esc50.csv")

print("WAV files :", len(audio_files))
print("CSV rows  :", len(meta))
print("Classes   :", meta["category"].nunique())

assert len(audio_files) == 2000
assert len(meta) == 2000
assert meta["category"].nunique() == 50

print("ESC-50 acquisition check: OK")

WAV files : 2000
CSV rows  : 2000
Classes   : 50
ESC-50 acquisition check: OK


In [7]:
CORE = {
    "crying_baby", "sneezing", "clapping", "breathing",
    "footsteps", "laughing", "brushing_teeth", "drinking_sipping",
    "door_wood_knock", "mouse_click", "keyboard_typing",
    "door_wood_creaks", "can_opening", "washing_machine",
    "vacuum_cleaner", "clock_alarm", "clock_tick", "glass_breaking",
}

TARGET = {
    "coughing": "COUGH",
    "snoring": "SNORE",
}

EXCLUDE = {
    "dog", "rooster", "pig", "cow", "frog",
    "cat", "hen", "insects", "sheep", "crow",
}

def esc_usage(category):
    if category in TARGET:
        return "target"
    if category in CORE:
        return "core"
    if category in EXCLUDE:
        return "exclude"
    return "extended"

meta["trpn_usage"] = meta["category"].map(esc_usage)

print(meta["trpn_usage"].value_counts())

trpn_usage
extended    800
core        720
exclude     400
target       80
Name: count, dtype: int64


#### manufest作成

In [21]:
from pathlib import Path
import pandas as pd
import wave
import hashlib

# ============================================================
# Path
# ============================================================

ESC_ROOT = Path(
    "/content/drive/MyDrive/TRON/AI/dataset/raw/esc50"
)

AUDIO_DIR = ESC_ROOT / "audio"
META_CSV = ESC_ROOT / "meta" / "esc50.csv"

OUT_DIR = Path(
    "/content/drive/MyDrive/TRON/AI/dataset/metadata/esc50"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV = OUT_DIR / "source_manifest.csv"


# ============================================================
# TRON project class definition
# ============================================================

CLASS_OTHER = 0
CLASS_SNORE = 1
CLASS_COUGH = 2


# ESC-50 -> TRON usage mapping

CORE = {
    # Human, non-speech sounds
    "crying_baby",
    "sneezing",
    "clapping",
    "breathing",
    "footsteps",
    "laughing",
    "brushing_teeth",
    "drinking_sipping",

    # Interior / domestic sounds
    "door_wood_knock",
    "mouse_click",
    "keyboard_typing",
    "door_wood_creaks",
    "can_opening",
    "washing_machine",
    "vacuum_cleaner",
    "clock_alarm",
    "clock_tick",
    "glass_breaking",
}

EXTENDED = {
    # Natural soundscapes & water sounds
    "rain",
    "sea_waves",
    "crackling_fire",
    "crickets",
    "chirping_birds",
    "water_drops",
    "wind",
    "pouring_water",
    "toilet_flush",
    "thunderstorm",

    # Exterior / urban noises
    "helicopter",
    "chainsaw",
    "siren",
    "car_horn",
    "engine",
    "train",
    "church_bells",
    "airplane",
    "fireworks",
    "hand_saw",
}

EXCLUDE = {
    # Animals
    "dog",
    "rooster",
    "pig",
    "cow",
    "frog",
    "cat",
    "hen",
    "insects",
    "sheep",
    "crow",
}

TARGET = {
    "snoring": (CLASS_SNORE, "SNORE"),
    "coughing": (CLASS_COUGH, "COUGH"),
}


# ============================================================
# Utility
# ============================================================

def get_wav_info(path):
    with wave.open(str(path), "rb") as wf:
        sr = wf.getframerate()
        channels = wf.getnchannels()
        frames = wf.getnframes()
        sample_width = wf.getsampwidth()

    return {
        "original_sr": sr,
        "original_channels": channels,
        "sample_width_bytes": sample_width,
        "duration_sec": frames / sr,
    }


def classify_category(category):
    """
    Return:
        usage_group
        class_id
        class_name
    """

    if category in TARGET:
        class_id, class_name = TARGET[category]
        return "target", class_id, class_name

    if category in CORE:
        return "core", CLASS_OTHER, "OTHER"

    if category in EXTENDED:
        return "extended", CLASS_OTHER, "OTHER"

    if category in EXCLUDE:
        return "exclude", pd.NA, "UNUSED"

    raise ValueError(
        f"Unknown ESC-50 category: {category}"
    )


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


# ============================================================
# Load official metadata
# ============================================================

meta = pd.read_csv(META_CSV)

print("Metadata rows :", len(meta))
print("Classes       :", meta["category"].nunique())


# ============================================================
# Build manifest
# ============================================================

rows = []

for _, row in meta.iterrows():

    filename = row["filename"]
    audio_path = AUDIO_DIR / filename

    usage_group, class_id, class_name = classify_category(
        row["category"]
    )

    if audio_path.exists():
        info = get_wav_info(audio_path)
        availability = "available"
        size_bytes = audio_path.stat().st_size

    else:
        info = {
            "original_sr": None,
            "original_channels": None,
            "sample_width_bytes": None,
            "duration_sec": None,
        }

        availability = "missing"
        size_bytes = None

    # ESC-10だけはCC BY 3.0
    # それ以外のESC-50はCC BY-NC 3.0
    if bool(row["esc10"]):
        license_name = "CC BY 3.0"
    else:
        license_name = "CC BY-NC 3.0"

    source_id = f"esc50:{filename}"

    # src_fileを使って、
    # 同じ元Freesound音源由来のfragmentを同一groupにする
    provenance_group_id = (
        f"esc50:src:{row['src_file']}"
    )

    rows.append({

        # ----------------------------------------------------
        # Common manifest fields
        # ----------------------------------------------------

        "source_id": source_id,

        "source_dataset": "ESC-50",

        "dataset_version": "ESC-50-master",

        "source_recording_id": filename,

        "subject_id": pd.NA,

        "session_id": pd.NA,

        "original_label": row["category"],

        "class_id": class_id,

        "class_name": class_name,

        "usage_group": usage_group,

        "relative_path":
            audio_path.relative_to(ESC_ROOT).as_posix(),

        "original_sr": info["original_sr"],

        "original_channels":
            info["original_channels"],

        "sample_width_bytes":
            info["sample_width_bytes"],

        "duration_sec":
            info["duration_sec"],

        "size_bytes": size_bytes,

        "quality": pd.NA,

        "license": license_name,

        "source_url":
            "https://github.com/karolpiczak/ESC-50",

        "provenance_group_id":
            provenance_group_id,

        "availability":
            availability,

        # AI-05まで未割当
        "split": "UNASSIGNED",

        "notes": "",


        # ----------------------------------------------------
        # ESC-50 specific metadata
        # ----------------------------------------------------

        "esc50_fold": row["fold"],

        "esc50_target": row["target"],

        "esc50_category": row["category"],

        "esc10": row["esc10"],

        "esc50_src_file": row["src_file"],

        "esc50_take": row["take"],
    })


manifest = pd.DataFrame(rows)

# nullable integerにする
manifest["class_id"] = (
    manifest["class_id"]
    .astype("Int64")
)

manifest.to_csv(
    OUT_CSV,
    index=False
)

print()
print("Saved:")
print(OUT_CSV)

Metadata rows : 2000
Classes       : 50

Saved:
/content/drive/MyDrive/TRON/AI/dataset/metadata/esc50/source_manifest.csv


In [22]:
print("=== Basic check ===")

print("Rows       :", len(manifest))
print("Files OK   :", (manifest["availability"] == "available").sum())
print("Categories :", manifest["esc50_category"].nunique())

print()
print("=== Usage group ===")

print(
    manifest["usage_group"]
    .value_counts()
)

print()
print("=== TRON class ===")

print(
    manifest["class_name"]
    .value_counts(dropna=False)
)

print()
print("=== Audio format ===")

print(
    manifest[
        [
            "original_sr",
            "original_channels",
            "sample_width_bytes",
            "duration_sec",
        ]
    ]
    .drop_duplicates()
)

print()
print("=== Provenance ===")

print(
    "Unique source files:",
    manifest["source_id"].nunique()
)

print(
    "Unique provenance groups:",
    manifest["provenance_group_id"].nunique()
)


# ============================================================
# Assertions
# ============================================================

assert len(manifest) == 2000

assert manifest["source_id"].is_unique

assert manifest["esc50_category"].nunique() == 50

assert (
    manifest["availability"] == "available"
).all()

counts = (
    manifest["usage_group"]
    .value_counts()
    .to_dict()
)

assert counts["core"] == 720
assert counts["extended"] == 800
assert counts["exclude"] == 400
assert counts["target"] == 80

assert (
    manifest[
        manifest["usage_group"] == "target"
    ]["class_name"]
    .value_counts()
    .to_dict()
    == {
        "COUGH": 40,
        "SNORE": 40,
    }
)

print()
print("ESC-50 source manifest check: OK")

=== Basic check ===
Rows       : 2000
Files OK   : 2000
Categories : 50

=== Usage group ===
usage_group
extended    800
core        720
exclude     400
target       80
Name: count, dtype: int64

=== TRON class ===
class_name
OTHER     1520
UNUSED     400
COUGH       40
SNORE       40
Name: count, dtype: int64

=== Audio format ===
   original_sr  original_channels  sample_width_bytes  duration_sec
0        44100                  1                   2           5.0

=== Provenance ===
Unique source files: 2000
Unique provenance groups: 1524

ESC-50 source manifest check: OK


## Coswaraのデータ取得

#### 単一日程取得

In [8]:
COSWARA_OUT = Path(
    "/content/drive/MyDrive/TRON/AI/dataset/raw/coswara"
)
COSWARA_META = Path(
    "/content/drive/MyDrive/TRON/AI/dataset/metadata/coswara"
)

COSWARA_OUT.mkdir(parents=True, exist_ok=True)
COSWARA_META.mkdir(parents=True, exist_ok=True)

print(COSWARA_OUT)

/content/drive/MyDrive/TRON/AI/dataset/raw/coswara


In [9]:
!rm -rf /content/Coswara-Data

!git clone \
    --filter=blob:none \
    --sparse \
    https://github.com/iiscleap/Coswara-Data.git \
    /content/Coswara-Data

Cloning into '/content/Coswara-Data'...
remote: Enumerating objects: 437, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 437 (delta 21), reused 9 (delta 9), pack-reused 412 (from 3)
Receiving objects: 100% (437/437), 80.64 KiB | 3.51 MiB/s, done.
Resolving deltas: 100% (154/154), done.
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 6 (delta 0), reused 0 (delta 0), pack-reused 1 (from 1)
Receiving objects: 100% (6/6), 107.01 KiB | 6.69 MiB/s, done.


In [10]:
import subprocess

REPO = Path("/content/Coswara-Data")

commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True
).strip()

print("Coswara commit:", commit)

(COSWARA_META / "source_commit.txt").write_text(
    commit + "\n",
    encoding="utf-8"
)

Coswara commit: 4942c97e31de7180a93d17f2e7530a9c543cfd50


41

In [11]:
!cd /content/Coswara-Data && git sparse-checkout set 20200413

remote: Enumerating objects: 5, done.
remote: Total 5 (delta 0), reused 0 (delta 0), pack-reused 5 (from 1)
Receiving objects: 100% (5/5), 348.53 MiB | 34.29 MiB/s, done.
Updating files: 100% (5/5), done.


In [12]:
!ls -lh /content/Coswara-Data/20200413

total 349M
-rw-r--r-- 1 root root 7.7K Sep 12 10:31 20200413.csv
-rw-r--r-- 1 root root  95M Sep 12 10:31 20200413.tar.gz.aa
-rw-r--r-- 1 root root  95M Sep 12 10:31 20200413.tar.gz.ab
-rw-r--r-- 1 root root  95M Sep 12 10:31 20200413.tar.gz.ac
-rw-r--r-- 1 root root  64M Sep 12 10:31 20200413.tar.gz.ad


In [13]:
import shutil
import subprocess

DATE = "20200413"

date_dir = REPO / DATE
tmp_dir = Path("/content/coswara_extract")

if tmp_dir.exists():
    shutil.rmtree(tmp_dir)

tmp_dir.mkdir()

cmd = (
    f"cat {date_dir}/*.tar.gz.* "
    f"| tar -xz -f - -C {tmp_dir}"
)

subprocess.run(cmd, shell=True, check=True)

print("Extraction complete")

Extraction complete


In [14]:
wanted = {
    "cough-heavy.wav",
    "cough-shallow.wav",
}

copied = []

for wav in tmp_dir.rglob("*.wav"):
    if wav.name not in wanted:
        continue

    subject_id = wav.parent.name

    dst_dir = COSWARA_OUT / subject_id
    dst_dir.mkdir(parents=True, exist_ok=True)

    dst = dst_dir / wav.name
    shutil.copy2(wav, dst)

    copied.append({
        "date": DATE,
        "subject_id": subject_id,
        "recording": wav.name,
        "path": str(dst),
    })

print("Copied files:", len(copied))
print("Subjects:", len({x["subject_id"] for x in copied}))

Copied files: 152
Subjects: 76


In [15]:
shutil.rmtree(tmp_dir)
print("Temporary extracted data removed.")

Temporary extracted data removed.


#### 全日程取得

In [16]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import shutil
import re
import json
import wave
import pandas as pd

# GitHubの一時clone先
REPO = Path("/content/Coswara-Data")

# 一時展開先
WORK_ROOT = Path("/content/coswara_work")

# Google Drive
RAW_ROOT = Path(
    "/content/drive/MyDrive/TRON/AI/dataset/raw/coswara"
)

META_ROOT = Path(
    "/content/drive/MyDrive/TRON/AI/dataset/metadata/coswara"
)

DATE_CSV_ROOT = META_ROOT / "date_csv"
COMPLETE_ROOT = META_ROOT / "completed_dates"

for p in [
    RAW_ROOT,
    META_ROOT,
    DATE_CSV_ROOT,
    COMPLETE_ROOT,
    WORK_ROOT,
]:
    p.mkdir(parents=True, exist_ok=True)

print("RAW :", RAW_ROOT)
print("META:", META_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RAW : /content/drive/MyDrive/TRON/AI/dataset/raw/coswara
META: /content/drive/MyDrive/TRON/AI/dataset/metadata/coswara


In [17]:
REPO_URL = "https://github.com/iiscleap/Coswara-Data.git"

if not REPO.exists():
    subprocess.run([
        "git",
        "clone",
        "--depth", "1",
        "--filter=blob:none",
        "--sparse",
        REPO_URL,
        str(REPO),
    ], check=True)

elif not (REPO / ".git").exists():
    raise RuntimeError(
        f"{REPO} already exists but is not a Git repository."
    )

commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True
).strip()

print("Coswara commit:", commit)

# 再現性のためcommitを保存
(META_ROOT / "source_commit.txt").write_text(
    commit + "\n",
    encoding="utf-8"
)

# YYYYMMDD形式のトップレベルdirectoryだけ取得
top_dirs = subprocess.check_output(
    [
        "git",
        "-C", str(REPO),
        "ls-tree",
        "-d",
        "--name-only",
        "HEAD",
    ],
    text=True,
).splitlines()

dates = sorted(
    d for d in top_dirs
    if re.fullmatch(r"\d{8}", d)
)

print("Date folders:", len(dates))
print(dates)

Coswara commit: 4942c97e31de7180a93d17f2e7530a9c543cfd50
Date folders: 43
['20200413', '20200415', '20200416', '20200417', '20200418', '20200419', '20200424', '20200430', '20200502', '20200504', '20200505', '20200525', '20200604', '20200707', '20200720', '20200803', '20200814', '20200820', '20200824', '20200901', '20200911', '20200919', '20200930', '20201012', '20201031', '20201130', '20201221', '20210206', '20210406', '20210419', '20210426', '20210507', '20210523', '20210603', '20210618', '20210630', '20210714', '20210816', '20210830', '20210914', '20210930', '20220116', '20220224']


In [18]:
def export_git_file(relative_path, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    with destination.open("wb") as f:
        subprocess.run(
            [
                "git",
                "-C", str(REPO),
                "show",
                f"HEAD:{relative_path}",
            ],
            stdout=f,
            check=True,
        )


# dataset全体のmetadata
for filename in [
    "LICENSE.md",
    "README.md",
    "combined_data.csv",
    "csv_labels_legend.json",
]:
    export_git_file(
        filename,
        META_ROOT / filename
    )


# quality annotation
annotation_files = subprocess.check_output(
    [
        "git",
        "-C", str(REPO),
        "ls-tree",
        "-r",
        "--name-only",
        "HEAD",
        "annotations",
    ],
    text=True,
).splitlines()

for relative_path in annotation_files:
    export_git_file(
        relative_path,
        META_ROOT / relative_path
    )

print("Metadata exported.")

Metadata exported.


In [19]:
WANTED_FILES = {
    "cough-heavy.wav",
    "cough-shallow.wav",
}

LOG_PATH = META_ROOT / "acquisition_log.csv"


def get_wav_info(path):
    """WAV headerから基本情報を取得する。"""
    try:
        with wave.open(str(path), "rb") as wf:
            channels = wf.getnchannels()
            sample_rate = wf.getframerate()
            frames = wf.getnframes()
            sample_width = wf.getsampwidth()

        duration = frames / sample_rate

        return {
            "original_sr": sample_rate,
            "original_channels": channels,
            "sample_width_bytes": sample_width,
            "duration_sec": duration,
        }

    except Exception:
        return {
            "original_sr": None,
            "original_channels": None,
            "sample_width_bytes": None,
            "duration_sec": None,
        }


def update_log(rows):
    """取得ログをsource_recording_id単位で更新する。"""
    if not rows:
        return

    new_df = pd.DataFrame(rows)

    if LOG_PATH.exists():
        old_df = pd.read_csv(LOG_PATH)
        df = pd.concat(
            [old_df, new_df],
            ignore_index=True
        )
    else:
        df = new_df

    df = (
        df
        .drop_duplicates(
            subset=["source_recording_id"],
            keep="last"
        )
        .sort_values([
            "collection_date",
            "subject_id",
            "recording_type",
        ])
    )

    df.to_csv(
        LOG_PATH,
        index=False
    )


def process_date(date):
    """1収集日を取得→展開→coughのみDrive保存する。"""

    complete_marker = COMPLETE_ROOT / f"{date}.json"

    # 完了済みならskip
    if complete_marker.exists():
        print(f"[SKIP] {date}: already completed")
        return

    print()
    print("=" * 60)
    print(f"Processing {date}")
    print("=" * 60)

    # この日付だけcheckout
    subprocess.run(
        [
            "git",
            "-C", str(REPO),
            "sparse-checkout",
            "set",
            date,
        ],
        check=True,
    )

    date_dir = REPO / date

    if not date_dir.exists():
        raise RuntimeError(
            f"Date directory not found: {date_dir}"
        )

    # 日付ごとのmetadata CSVも保存
    source_csv = date_dir / f"{date}.csv"

    if source_csv.exists():
        shutil.copy2(
            source_csv,
            DATE_CSV_ROOT / source_csv.name
        )

    # 分割tar.gzを確認
    shards = sorted(
        date_dir.glob(f"{date}.tar.gz.*")
    )

    if not shards:
        raise RuntimeError(
            f"No archive shards found for {date}"
        )

    print("Archive shards:", len(shards))

    # 一時展開先
    tmp_dir = WORK_ROOT / date

    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)

    tmp_dir.mkdir(parents=True)

    try:
        # 公式extract_data.pyと同様に
        # 分割archiveを結合しながら展開
        cmd = (
            f'cat "{date_dir}/{date}.tar.gz."* '
            f'| tar -xz -C "{tmp_dir}"'
        )

        subprocess.run(
            ["bash", "-lc", cmd],
            check=True,
        )

        # cough 2種類だけ抽出
        cough_files = sorted(
            p for p in tmp_dir.rglob("*.wav")
            if p.name in WANTED_FILES
        )

        print("Found cough files:", len(cough_files))

        date_output = RAW_ROOT / date
        date_output.mkdir(
            parents=True,
            exist_ok=True
        )

        rows = []
        subjects = set()

        copied = 0
        existing = 0
        replaced = 0

        for src in cough_files:
            # 親folder名がparticipant ID
            subject_id = src.parent.name
            subjects.add(subject_id)

            subject_dir = date_output / subject_id
            subject_dir.mkdir(
                parents=True,
                exist_ok=True
            )

            dst = subject_dir / src.name

            if dst.exists():
                if dst.stat().st_size == src.stat().st_size:
                    status = "existing"
                    existing += 1
                else:
                    shutil.copy2(src, dst)
                    status = "replaced"
                    replaced += 1
            else:
                shutil.copy2(src, dst)
                status = "copied"
                copied += 1

            info = get_wav_info(dst)

            rows.append({
                "source_dataset": "Coswara",
                "dataset_version": commit,
                "collection_date": date,

                "subject_id": subject_id,
                "provenance_group_id": subject_id,

                "source_recording_id":
                    f"coswara:{date}:{subject_id}:{src.name}",

                "recording_type": src.stem,
                "original_label": src.stem,

                "class_id": 2,
                "class_name": "COUGH",

                "relative_path":
                    dst.relative_to(RAW_ROOT).as_posix(),

                "size_bytes": dst.stat().st_size,

                "original_sr":
                    info["original_sr"],

                "original_channels":
                    info["original_channels"],

                "sample_width_bytes":
                    info["sample_width_bytes"],

                "duration_sec":
                    info["duration_sec"],

                "license": "CC BY 4.0",

                "status": status,
            })

        # Drive上のlogを更新
        update_log(rows)

        summary = {
            "collection_date": date,
            "subjects": len(subjects),
            "recordings": len(cough_files),
            "copied": copied,
            "existing": existing,
            "replaced": replaced,
            "dataset_commit": commit,
        }

        complete_marker.write_text(
            json.dumps(
                summary,
                indent=2
            ),
            encoding="utf-8",
        )

        print(
            f"Subjects : {len(subjects)}"
        )
        print(
            f"Recordings: {len(cough_files)}"
        )
        print(
            f"Copied   : {copied}"
        )
        print(
            f"Existing : {existing}"
        )
        print(
            f"Replaced : {replaced}"
        )

    finally:
        # 全9種類を展開した一時領域は必ず削除
        if tmp_dir.exists():
            shutil.rmtree(tmp_dir)

In [20]:
failed_dates = []

for i, date in enumerate(dates, start=1):
    print(
        f"\n[{i}/{len(dates)}] {date}"
    )

    try:
        process_date(date)

    except Exception as e:
        print(
            f"[ERROR] {date}: {e}"
        )
        failed_dates.append({
            "date": date,
            "error": repr(e),
        })


print()
print("=" * 60)
print("All dates processed.")
print("Failed dates:", len(failed_dates))

for x in failed_dates:
    print(
        x["date"],
        x["error"]
    )


[1/43] 20200413

Processing 20200413
Archive shards: 4
Found cough files: 152
Subjects : 76
Recordings: 152
Copied   : 0
Existing : 152
Replaced : 0

[2/43] 20200415

Processing 20200415
Archive shards: 9
Found cough files: 322
Subjects : 161
Recordings: 322
Copied   : 322
Existing : 0
Replaced : 0

[3/43] 20200416

Processing 20200416
Archive shards: 11
Found cough files: 393
Subjects : 197
Recordings: 393
Copied   : 393
Existing : 0
Replaced : 0

[4/43] 20200417

Processing 20200417
Archive shards: 10
Found cough files: 336
Subjects : 168
Recordings: 336
Copied   : 336
Existing : 0
Replaced : 0

[5/43] 20200418

Processing 20200418
Archive shards: 3
Found cough files: 92
Subjects : 46
Recordings: 92
Copied   : 92
Existing : 0
Replaced : 0

[6/43] 20200419

Processing 20200419
Archive shards: 2
Found cough files: 64
Subjects : 32
Recordings: 64
Copied   : 64
Existing : 0
Replaced : 0

[7/43] 20200424

Processing 20200424
Archive shards: 2
Found cough files: 56
Subjects : 28
Recording

In [ ]:
log = pd.read_csv(
    META_ROOT / "acquisition_log.csv"
)

print("Total recordings:", len(log))
print(
    "Unique subjects:",
    log["subject_id"].nunique()
)
print(
    "Collection dates:",
    log["collection_date"].nunique()
)

print()
print(
    log["recording_type"].value_counts()
)

print()
print(
    log[[
        "original_sr",
        "original_channels",
        "duration_sec",
    ]].describe()
)

assert log["source_recording_id"].is_unique

print()
print("Coswara acquisition check: OK")

In [ ]:
shutil.rmtree(REPO)
shutil.rmtree(WORK_ROOT)

#### manufest作成

In [28]:
from pathlib import Path
import pandas as pd
import wave

# ============================================================
# Path
# ============================================================

COSWARA_ROOT = Path(
    "/content/drive/MyDrive/TRON/AI/dataset/raw/coswara"
)

META_ROOT = Path(
    "/content/drive/MyDrive/TRON/AI/dataset/metadata/coswara"
)

OUT_CSV = META_ROOT / "source_manifest.csv"

META_ROOT.mkdir(parents=True, exist_ok=True)

# 取得時に保存したcommit hash
COMMIT_FILE = META_ROOT / "source_commit.txt"

if COMMIT_FILE.exists():
    DATASET_VERSION = COMMIT_FILE.read_text(
        encoding="utf-8"
    ).strip()
else:
    DATASET_VERSION = "UNKNOWN"

print("Dataset version:", DATASET_VERSION)


# ============================================================
# TRON class
# ============================================================

CLASS_COUGH = 2


# ============================================================
# WAV utility
# ============================================================

def get_wav_info(path):
    try:
        with wave.open(str(path), "rb") as wf:
            sr = wf.getframerate()
            channels = wf.getnchannels()
            frames = wf.getnframes()
            sample_width = wf.getsampwidth()

        return {
            "original_sr": sr,
            "original_channels": channels,
            "sample_width_bytes": sample_width,
            "duration_sec": frames / sr,
        }

    except Exception:
        return {
            "original_sr": None,
            "original_channels": None,
            "sample_width_bytes": None,
            "duration_sec": None,
        }


# ============================================================
# Quality annotation
# ============================================================

QUALITY_MAP = {
    0: "bad",
    1: "good",
    2: "excellent",
}

quality_lookup = {}

annotation_dir = META_ROOT / "annotations"

annotation_files = {
    "cough-heavy": annotation_dir / "cough-heavy_labels.csv",
    "cough-shallow": annotation_dir / "cough-shallow_labels.csv",
}


for recording_type, path in annotation_files.items():

    df = pd.read_csv(path)

    # " QUALITY" のような余分な空白を除去
    df.columns = df.columns.str.strip()

    suffix = f"_{recording_type}"

    for _, row in df.iterrows():

        filename = str(row["FILENAME"]).strip()

        if not filename.endswith(suffix):
            print(
                f"[WARNING] unexpected filename: {filename}"
            )
            continue

        subject_id = filename.removesuffix(suffix)

        quality_value = int(row["QUALITY"])

        quality_lookup[
            (subject_id, recording_type)
        ] = quality_value


print()
print("Quality labels loaded:", len(quality_lookup))


# ============================================================
# Build manifest
# ============================================================

rows = []

date_dirs = sorted(
    p for p in COSWARA_ROOT.iterdir()
    if p.is_dir() and p.name.isdigit() and len(p.name) == 8
)

for date_dir in date_dirs:

    collection_date = date_dir.name

    subject_dirs = sorted(
        p for p in date_dir.iterdir()
        if p.is_dir()
    )

    for subject_dir in subject_dirs:

        subject_id = subject_dir.name

        for recording_type in [
            "cough-heavy",
            "cough-shallow",
        ]:

            wav_path = subject_dir / f"{recording_type}.wav"

            if not wav_path.exists():
                continue

            info = get_wav_info(wav_path)

            quality_value = quality_lookup.get(
                (subject_id, recording_type)
            )

            if quality_value in QUALITY_MAP:
                quality = QUALITY_MAP[quality_value]
            else:
                quality = pd.NA

            source_recording_id = (
                f"{collection_date}:"
                f"{subject_id}:"
                f"{recording_type}"
            )

            rows.append({

                # ---------------------------------------------
                # Common fields
                # ---------------------------------------------

                "source_id":
                    f"coswara:{source_recording_id}",

                "source_dataset":
                    "Coswara",

                "dataset_version":
                    DATASET_VERSION,

                "source_recording_id":
                    source_recording_id,

                "subject_id":
                    subject_id,

                "session_id":
                    collection_date,

                "original_label":
                    recording_type,

                "class_id":
                    CLASS_COUGH,

                "class_name":
                    "COUGH",

                # Coswaraは主COUGHデータなのでtarget
                "usage_group":
                    "target",

                "relative_path":
                    wav_path.relative_to(
                        COSWARA_ROOT
                    ).as_posix(),

                "original_sr":
                    info["original_sr"],

                "original_channels":
                    info["original_channels"],

                "sample_width_bytes":
                    info["sample_width_bytes"],

                "duration_sec":
                    info["duration_sec"],

                "size_bytes":
                    wav_path.stat().st_size,

                "quality":
                    quality,

                "license":
                    "CC BY 4.0",

                "source_url":
                    "https://github.com/iiscleap/Coswara-Data",

                # 同一人物は必ず同じsplitにする
                "provenance_group_id":
                    f"coswara:subject:{subject_id}",

                "availability":
                    "available",

                "split":
                    "UNASSIGNED",

                "notes":
                    "",


                # ---------------------------------------------
                # Coswara specific
                # ---------------------------------------------

                "collection_date":
                    collection_date,

                "recording_type":
                    recording_type,

                "coswara_quality_value":
                    quality_value,
            })


manifest = pd.DataFrame(rows)

manifest["class_id"] = (
    manifest["class_id"]
    .astype("Int64")
)

manifest["coswara_quality_value"] = (
    manifest["coswara_quality_value"]
    .astype("Int64")
)

manifest.to_csv(
    OUT_CSV,
    index=False
)

print()
print("Saved:")
print(OUT_CSV)

Dataset version: 4942c97e31de7180a93d17f2e7530a9c543cfd50

Quality labels loaded: 5491

Saved:
/content/drive/MyDrive/TRON/AI/dataset/metadata/coswara/source_manifest.csv


In [24]:
print("=== Basic check ===")

print("Rows             :", len(manifest))
print("Unique subjects  :", manifest["subject_id"].nunique())
print("Collection dates :", manifest["collection_date"].nunique())
print("Files OK         :", (manifest["availability"] == "available").sum())

print()
print("=== Recording type ===")

print(
    manifest["recording_type"]
    .value_counts()
)

print()
print("=== Quality ===")

print(
    manifest["quality"]
    .value_counts(dropna=False)
)

print()
print("=== Audio format ===")

print(
    manifest[
        [
            "original_sr",
            "original_channels",
            "sample_width_bytes",
        ]
    ]
    .value_counts()
)

print()
print("=== Duration ===")

print(
    manifest["duration_sec"]
    .describe()
)

print()
print("=== Provenance ===")

print(
    "Unique source IDs:",
    manifest["source_id"].nunique()
)

print(
    "Unique provenance groups:",
    manifest["provenance_group_id"].nunique()
)


# ============================================================
# Assertions
# ============================================================

assert manifest["source_id"].is_unique

assert (
    manifest["class_id"] == 2
).all()

assert (
    manifest["class_name"] == "COUGH"
).all()

assert (
    manifest["usage_group"] == "target"
).all()

assert (
    manifest["availability"] == "available"
).all()

assert (
    manifest["split"] == "UNASSIGNED"
).all()

# subject単位でprovenanceが一意であること
assert (
    manifest.groupby("subject_id")[
        "provenance_group_id"
    ].nunique().max()
    == 1
)

print()
print("Coswara source manifest check: OK")

=== Basic check ===
Rows             : 5491
Unique subjects  : 2746
Collection dates : 43
Files OK         : 5491

=== Recording type ===
recording_type
cough-heavy      2746
cough-shallow    2745
Name: count, dtype: int64

=== Quality ===
quality
<NA>    5491
Name: count, dtype: int64

=== Audio format ===
original_sr  original_channels  sample_width_bytes
48000        1                  2                     5039
44100        1                  2                      386
16000        1                  2                       60
192000       1                  2                        4
8000         1                  2                        2
Name: count, dtype: int64

=== Duration ===
count    5491.000000
mean        5.508352
std         2.649967
min         0.000000
25%         3.840000
50%         5.205333
75%         6.741333
max        30.037333
Name: duration_sec, dtype: float64

=== Provenance ===
Unique source IDs: 5491
Unique provenance groups: 2746

Coswara source manifes

In [29]:
print("=== Quality ===")

print(
    manifest["quality"]
    .value_counts(dropna=False)
)

print()

print(
    manifest["coswara_quality_value"]
    .value_counts(dropna=False)
)

print()

missing_quality = manifest[
    manifest["quality"].isna()
]

print(
    "Missing quality:",
    len(missing_quality)
)

assert len(manifest) == 5491
assert len(missing_quality) == 0

print(
    "Coswara quality annotation check: OK"
)

=== Quality ===
quality
excellent    4559
good          541
bad           391
Name: count, dtype: int64

coswara_quality_value
2    4559
1     541
0     391
Name: count, dtype: Int64

Missing quality: 0
Coswara quality annotation check: OK


In [27]:
from pathlib import Path
import pandas as pd

META_ROOT = Path(
    "/content/drive/MyDrive/TRON/AI/dataset/metadata/coswara"
)

for name in [
    "cough-heavy_labels.csv",
    "cough-shallow_labels.csv",
]:
    path = META_ROOT / "annotations" / name

    df = pd.read_csv(path)

    print("=" * 80)
    print(name)
    print("shape:", df.shape)
    print("columns:")
    print(df.columns.tolist())
    print()
    print(df.head(10).to_string())
    print()
    print(df.dtypes)

cough-heavy_labels.csv
shape: (2746, 2)
columns:
['FILENAME', ' QUALITY']

                                   FILENAME   QUALITY
0  vK2bLRNzllXNeyOMudnNSL5cfpG2_cough-heavy         2
1  bjA2KpSxneNskrLBeqi4bqoTDQl2_cough-heavy         2
2  htQzROl26OWQpIYFDzv11F79PLR2_cough-heavy         2
3  pW9mCAeWYiMoM7wW7riLvNRbYDO2_cough-heavy         2
4  L7S8iIPKgiO6QWLC3mGkROCMa0s1_cough-heavy         2
5  eP8gEM0KcBU6S5JpMdycX74KP3p2_cough-heavy         2
6  UiUUhL0PMjWVA6W2dKte1DCE6wG2_cough-heavy         0
7  QQbzKidDxHQ0eZFEL4uG68s1lc02_cough-heavy         2
8  cfZUmZvsecYVKw5eSi4qIhfJVpI3_cough-heavy         2
9  tiKv850hJFTmpCAvfJOmOOqkeUs1_cough-heavy         0

FILENAME    object
 QUALITY     int64
dtype: object
cough-shallow_labels.csv
shape: (2745, 2)
columns:
['FILENAME', ' QUALITY']

                                     FILENAME   QUALITY
0  vK2bLRNzllXNeyOMudnNSL5cfpG2_cough-shallow         2
1  bjA2KpSxneNskrLBeqi4bqoTDQl2_cough-shallow         2
2  htQzROl26OWQpIYFDzv11F79PLR2_c

In [26]:
for name in [
    "cough-heavy_labels.csv",
    "cough-shallow_labels.csv",
]:
    path = META_ROOT / "annotations" / name

    print("=" * 60)
    print(name)

    df = pd.read_csv(
        path,
        header=None
    )

    print("shape:", df.shape)
    print(df.head(10))

cough-heavy_labels.csv
shape: (2747, 2)
                                          0         1
0                                  FILENAME   QUALITY
1  vK2bLRNzllXNeyOMudnNSL5cfpG2_cough-heavy         2
2  bjA2KpSxneNskrLBeqi4bqoTDQl2_cough-heavy         2
3  htQzROl26OWQpIYFDzv11F79PLR2_cough-heavy         2
4  pW9mCAeWYiMoM7wW7riLvNRbYDO2_cough-heavy         2
5  L7S8iIPKgiO6QWLC3mGkROCMa0s1_cough-heavy         2
6  eP8gEM0KcBU6S5JpMdycX74KP3p2_cough-heavy         2
7  UiUUhL0PMjWVA6W2dKte1DCE6wG2_cough-heavy         0
8  QQbzKidDxHQ0eZFEL4uG68s1lc02_cough-heavy         2
9  cfZUmZvsecYVKw5eSi4qIhfJVpI3_cough-heavy         2
cough-shallow_labels.csv
shape: (2746, 2)
                                            0         1
0                                    FILENAME   QUALITY
1  vK2bLRNzllXNeyOMudnNSL5cfpG2_cough-shallow         2
2  bjA2KpSxneNskrLBeqi4bqoTDQl2_cough-shallow         2
3  htQzROl26OWQpIYFDzv11F79PLR2_cough-shallow         2
4  pW9mCAeWYiMoM7wW7riLvNRbYDO2_cough-shallo

## AudioSet Strong-Snoring のデータ取得

1. Strong-label TSV取得
1. Snoring MIDだけ抽出
1. source clip / event件数確認
1. YouTubeから現在取得可能な音声だけ取得
1. availabilityを記録
1. source_manifest.csv + events.csv生成

#### 前処理

In [30]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

RAW_ROOT = Path(
    "/content/drive/MyDrive/TRON/AI/dataset/raw/audioset_snoring"
)

META_ROOT = Path(
    "/content/drive/MyDrive/TRON/AI/dataset/metadata/audioset_snoring"
)

RAW_ROOT.mkdir(parents=True, exist_ok=True)
META_ROOT.mkdir(parents=True, exist_ok=True)

print(RAW_ROOT)
print(META_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/TRON/AI/dataset/raw/audioset_snoring
/content/drive/MyDrive/TRON/AI/dataset/metadata/audioset_snoring


In [32]:
import urllib.request

files = {
    "audioset_train_strong.tsv":
        "https://storage.googleapis.com/us_audioset/youtube_corpus/strong/audioset_train_strong.tsv",

    "audioset_eval_strong.tsv":
        "https://storage.googleapis.com/us_audioset/youtube_corpus/strong/audioset_eval_strong.tsv",

    "mid_to_display_name.tsv":
        "https://storage.googleapis.com/us_audioset/youtube_corpus/strong/mid_to_display_name.tsv",
}

for filename, url in files.items():

    dst = META_ROOT / filename

    if dst.exists():
        print("[SKIP]", filename)
        continue

    print("Downloading:", filename)

    urllib.request.urlretrieve(
        url,
        dst
    )

print("Metadata download complete.")

Downloading: audioset_train_strong.tsv
Downloading: audioset_eval_strong.tsv
Downloading: mid_to_display_name.tsv
Metadata download complete.


In [33]:
import pandas as pd

mid_map = pd.read_csv(
    META_ROOT / "mid_to_display_name.tsv",
    sep="\t",
    names=["mid", "display_name"],
)

snoring_row = mid_map[
    mid_map["display_name"] == "Snoring"
]

print(snoring_row)

          mid display_name
28  /m/01d3sd      Snoring


In [34]:
SNORING_MID = snoring_row.iloc[0]["mid"]

columns = [
    "clip_id",
    "event_start",
    "event_end",
    "mid",
]

train = pd.read_csv(
    META_ROOT / "audioset_train_strong.tsv",
    sep="\t",
    names=columns,
)

eval_df = pd.read_csv(
    META_ROOT / "audioset_eval_strong.tsv",
    sep="\t",
    names=columns,
)

train_snore = train[
    train["mid"] == SNORING_MID
].copy()

eval_snore = eval_df[
    eval_df["mid"] == SNORING_MID
].copy()

train_snore["official_split"] = "train"
eval_snore["official_split"] = "eval"

snore_events = pd.concat(
    [
        train_snore,
        eval_snore,
    ],
    ignore_index=True,
)

print("MID:", SNORING_MID)

print()
print("Train events:", len(train_snore))
print("Eval events :", len(eval_snore))
print("Total events:", len(snore_events))

print()
print(
    "Unique source clips:",
    snore_events["clip_id"].nunique()
)

/tmp/ipykernel_1897/3858814652.py:10: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(


MID: /m/01d3sd

Train events: 1158
Eval events : 158
Total events: 1316

Unique source clips: 465


/tmp/ipykernel_1897/3858814652.py:16: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  eval_df = pd.read_csv(


In [36]:
def parse_clip_id(clip_id):

    youtube_id, start_ms = clip_id.rsplit("_", 1)

    return pd.Series({
        "youtube_id": youtube_id,
        "clip_start_sec": int(start_ms) / 1000.0,
    })


parsed = snore_events["clip_id"].apply(
    parse_clip_id
)

snore_events = pd.concat(
    [
        snore_events,
        parsed,
    ],
    axis=1,
)

# 数値型へ明示変換
snore_events["event_start"] = pd.to_numeric(
    snore_events["event_start"],
    errors="coerce"
)

snore_events["event_end"] = pd.to_numeric(
    snore_events["event_end"],
    errors="coerce"
)

# 変換失敗チェック
print(snore_events[["event_start", "event_end"]].dtypes)

bad_rows = snore_events[
    snore_events["event_start"].isna()
    | snore_events["event_end"].isna()
]

print("Invalid time rows:", len(bad_rows))

snore_events["event_duration"] = (
    snore_events["event_end"]
    - snore_events["event_start"]
)

print(
    snore_events[
        [
            "clip_id",
            "youtube_id",
            "clip_start_sec",
            "event_start",
            "event_end",
            "event_duration",
        ]
    ].head()
)

event_start    float64
event_end      float64
dtype: object
Invalid time rows: 0
             clip_id   youtube_id   youtube_id  clip_start_sec  \
0  T98j2xSPmNI_10000  T98j2xSPmNI  T98j2xSPmNI            10.0   
1  T98j2xSPmNI_10000  T98j2xSPmNI  T98j2xSPmNI            10.0   
2  T98j2xSPmNI_10000  T98j2xSPmNI  T98j2xSPmNI            10.0   
3  T98j2xSPmNI_10000  T98j2xSPmNI  T98j2xSPmNI            10.0   
4  -NrpfrEGq5w_40000  -NrpfrEGq5w  -NrpfrEGq5w            40.0   

   clip_start_sec  event_start  event_end  event_duration  
0            10.0        0.000      0.381           0.381  
1            10.0        3.138      4.422           1.284  
2            10.0        5.113      6.330           1.217  
3            10.0        8.677     10.000           1.323  
4            40.0        0.505      0.725           0.220  


In [37]:
EVENTS_CSV = META_ROOT / "snoring_events_official.csv"

snore_events.to_csv(
    EVENTS_CSV,
    index=False
)

print(EVENTS_CSV)

/content/drive/MyDrive/TRON/AI/dataset/metadata/audioset_snoring/snoring_events_official.csv


In [38]:
source_clips = (
    snore_events[
        [
            "clip_id",
            "youtube_id",
            "clip_start_sec",
            "official_split",
        ]
    ]
    .drop_duplicates(subset=["clip_id"])
    .sort_values("clip_id")
    .reset_index(drop=True)
)

source_clips.to_csv(
    META_ROOT / "snoring_source_clips.csv",
    index=False
)

print(
    "Source clips:",
    len(source_clips)
)

Source clips: 465


#### 音声取得処理

465 clipsを順番に処理

→ yt-dlpでYouTube音源取得

→ clip_start_secから10秒切り出し

→ mono WAV保存

→ 成功/失敗をログ保存

→ 途中再開可能

In [39]:
!python -m pip install -U "yt-dlp[default]"

!yt-dlp --version
!ffmpeg -version | head -n 1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.4/53.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 26.5 MB/s eta 0:00:00
2026.08.19
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


In [50]:
from pathlib import Path
import pandas as pd
import shutil
import subprocess
import json
import time

RAW_ROOT = Path(
    "/content/drive/MyDrive/TRON/AI/dataset/raw/audioset_snoring"
)

META_ROOT = Path(
    "/content/drive/MyDrive/TRON/AI/dataset/metadata/audioset_snoring"
)

WORK_ROOT = Path(
    "/content/audioset_snoring_work"
)

RAW_ROOT.mkdir(parents=True, exist_ok=True)
META_ROOT.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

SOURCE_CSV = META_ROOT / "snoring_source_clips.csv"
LOG_CSV = META_ROOT / "acquisition_log.csv"

source_clips = pd.read_csv(SOURCE_CSV)

print("Source clips:", len(source_clips))
print(source_clips.head())

Source clips: 465
             clip_id   youtube_id youtube_id.1  clip_start_sec  \
0  -7xDzCwyAco_30000  -7xDzCwyAco  -7xDzCwyAco            30.0   
1  -81GPDp9IQY_20000  -81GPDp9IQY  -81GPDp9IQY            20.0   
2      -DexfVugZDE_0  -DexfVugZDE  -DexfVugZDE             0.0   
3  -NrpfrEGq5w_40000  -NrpfrEGq5w  -NrpfrEGq5w            40.0   
4      -NsUQ2A0ClQ_0  -NsUQ2A0ClQ  -NsUQ2A0ClQ             0.0   

   clip_start_sec.1 official_split  
0              30.0          train  
1              20.0          train  
2               0.0          train  
3              40.0          train  
4               0.0          train  


In [51]:
import shutil
import subprocess
import wave

MIN_VALID_DURATION = 9.9
MAX_VALID_DURATION = 10.1


def get_wav_duration(path):
    try:
        with wave.open(str(path), "rb") as wf:
            sr = wf.getframerate()
            frames = wf.getnframes()

        if sr <= 0:
            return None

        return frames / sr

    except Exception:
        return None


def download_audioset_clip(row):

    clip_id = str(row["clip_id"])
    youtube_id = str(row["youtube_id"])
    clip_start = float(row["clip_start_sec"])

    output_wav = RAW_ROOT / f"{clip_id}.wav"

    # ========================================================
    # 既存ファイルの検証
    # ========================================================

    if output_wav.exists():

        duration = get_wav_duration(output_wav)

        if (
            duration is not None
            and MIN_VALID_DURATION <= duration <= MAX_VALID_DURATION
        ):
            return {
                "clip_id": clip_id,
                "youtube_id": youtube_id,
                "clip_start_sec": clip_start,
                "status": "existing",
                "relative_path":
                    output_wav.relative_to(RAW_ROOT).as_posix(),
                "duration_sec": duration,
                "error": "",
            }

        # 短い・壊れている既存WAVは削除して再取得
        print(
            f"[REMOVE INVALID] {clip_id}: "
            f"duration={duration}"
        )

        output_wav.unlink()


    # ========================================================
    # 一時作業directory
    # ========================================================

    clip_work = WORK_ROOT / clip_id

    if clip_work.exists():
        shutil.rmtree(clip_work)

    clip_work.mkdir(parents=True)

    url = (
        f"https://www.youtube.com/watch?v={youtube_id}"
    )


    try:

        # ====================================================
        # YouTubeから音声取得
        # ====================================================

        output_template = str(
            clip_work / "source.%(ext)s"
        )

        cmd = [
            "yt-dlp",
            "--no-playlist",
            "-f", "bestaudio/best",
            "-o", output_template,
            url,
        ]

        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=180,
        )

        if result.returncode != 0:

            return {
                "clip_id": clip_id,
                "youtube_id": youtube_id,
                "clip_start_sec": clip_start,
                "status": "download_error",
                "relative_path": "",
                "duration_sec": None,
                "error": result.stderr[-1000:],
            }


        downloaded = [
            p for p in clip_work.iterdir()
            if p.is_file()
        ]

        if not downloaded:

            return {
                "clip_id": clip_id,
                "youtube_id": youtube_id,
                "clip_start_sec": clip_start,
                "status": "download_error",
                "relative_path": "",
                "duration_sec": None,
                "error": "Downloaded file not found.",
            }


        source_audio = max(
            downloaded,
            key=lambda p: p.stat().st_size
        )


        # ====================================================
        # AudioSetの10秒区間を切り出す
        # ====================================================

        ffmpeg_cmd = [
            "ffmpeg",
            "-y",
            "-hide_banner",
            "-loglevel", "error",

            "-ss", str(clip_start),
            "-i", str(source_audio),

            "-t", "10.0",

            "-c:a", "pcm_s16le",

            str(output_wav),
        ]

        ffmpeg_result = subprocess.run(
            ffmpeg_cmd,
            capture_output=True,
            text=True,
            timeout=120,
        )

        if ffmpeg_result.returncode != 0:

            if output_wav.exists():
                output_wav.unlink()

            return {
                "clip_id": clip_id,
                "youtube_id": youtube_id,
                "clip_start_sec": clip_start,
                "status": "ffmpeg_error",
                "relative_path": "",
                "duration_sec": None,
                "error": ffmpeg_result.stderr[-1000:],
            }


        # ====================================================
        # 出力WAV検証
        # ====================================================

        if (
            not output_wav.exists()
            or output_wav.stat().st_size <= 44
        ):

            if output_wav.exists():
                output_wav.unlink()

            return {
                "clip_id": clip_id,
                "youtube_id": youtube_id,
                "clip_start_sec": clip_start,
                "status": "invalid_output",
                "relative_path": "",
                "duration_sec": None,
                "error": "Output WAV is empty or invalid.",
            }


        duration = get_wav_duration(
            output_wav
        )


        if duration is None:

            output_wav.unlink()

            return {
                "clip_id": clip_id,
                "youtube_id": youtube_id,
                "clip_start_sec": clip_start,
                "status": "invalid_output",
                "relative_path": "",
                "duration_sec": None,
                "error": "Could not read WAV duration.",
            }


        # ====================================================
        # ★ 10秒未満はavailableにしない
        # ====================================================

        if not (
            MIN_VALID_DURATION
            <= duration
            <= MAX_VALID_DURATION
        ):

            output_wav.unlink()

            return {
                "clip_id": clip_id,
                "youtube_id": youtube_id,
                "clip_start_sec": clip_start,
                "status": "short_clip",
                "relative_path": "",
                "duration_sec": duration,
                "error":
                    f"Expected ~10 s AudioSet excerpt, "
                    f"but obtained {duration:.3f} s.",
            }


        # ====================================================
        # ここまで通ったものだけavailable
        # ====================================================

        return {
            "clip_id": clip_id,
            "youtube_id": youtube_id,
            "clip_start_sec": clip_start,
            "status": "available",
            "relative_path":
                output_wav.relative_to(RAW_ROOT).as_posix(),
            "duration_sec": duration,
            "error": "",
        }


    except subprocess.TimeoutExpired:

        if output_wav.exists():
            output_wav.unlink()

        return {
            "clip_id": clip_id,
            "youtube_id": youtube_id,
            "clip_start_sec": clip_start,
            "status": "timeout",
            "relative_path": "",
            "duration_sec": None,
            "error": "Process timeout.",
        }


    except Exception as e:

        if output_wav.exists():
            output_wav.unlink()

        return {
            "clip_id": clip_id,
            "youtube_id": youtube_id,
            "clip_start_sec": clip_start,
            "status": "unexpected_error",
            "relative_path": "",
            "duration_sec": None,
            "error": repr(e),
        }


    finally:

        if clip_work.exists():
            shutil.rmtree(
                clip_work,
                ignore_errors=True
            )

In [52]:
test_results = []

for _, row in source_clips.head(5).iterrows():

    print(
        "Downloading:",
        row["clip_id"]
    )

    result = download_audioset_clip(row)

    test_results.append(result)

    print(
        result["status"],
        result["error"][:200]
    )

    # YouTubeへ連続アクセスしすぎない
    time.sleep(1)

Downloading: -7xDzCwyAco_30000
existing 
Downloading: -81GPDp9IQY_20000
download_error WARNING: [youtube] No supported JavaScript runtime could be found. Only deno is enabled by default; to use another runtime add  --js-runtimes RUNTIME[:PATH]  to your command/config. YouTube extraction
Downloading: -DexfVugZDE_0
[REMOVE INVALID] -DexfVugZDE_0: duration=6.710566893424036
short_clip Expected ~10 s AudioSet excerpt, but obtained 6.711 s.
Downloading: -NrpfrEGq5w_40000
existing 
Downloading: -NsUQ2A0ClQ_0
existing 


In [53]:
test_df = pd.DataFrame(test_results)

print(
    test_df[
        [
            "clip_id",
            "status",
            "relative_path",
        ]
    ]
)

             clip_id          status          relative_path
0  -7xDzCwyAco_30000        existing  -7xDzCwyAco_30000.wav
1  -81GPDp9IQY_20000  download_error                       
2      -DexfVugZDE_0      short_clip                       
3  -NrpfrEGq5w_40000        existing  -NrpfrEGq5w_40000.wav
4      -NsUQ2A0ClQ_0        existing      -NsUQ2A0ClQ_0.wav


In [54]:
import wave

def inspect_wav(path):
    with wave.open(str(path), "rb") as wf:

        sr = wf.getframerate()
        channels = wf.getnchannels()
        frames = wf.getnframes()
        width = wf.getsampwidth()

    return {
        "sample_rate": sr,
        "channels": channels,
        "sample_width": width,
        "frames": frames,
        "duration":
            frames / sr if sr else None,
    }


for result in test_results:

    if result["status"] not in {
        "available",
        "existing",
    }:
        continue

    path = RAW_ROOT / result["relative_path"]

    print(
        result["clip_id"],
        inspect_wav(path)
    )

-7xDzCwyAco_30000 {'sample_rate': 44100, 'channels': 2, 'sample_width': 2, 'frames': 441000, 'duration': 10.0}
-NrpfrEGq5w_40000 {'sample_rate': 44100, 'channels': 2, 'sample_width': 2, 'frames': 441000, 'duration': 10.0}
-NsUQ2A0ClQ_0 {'sample_rate': 48000, 'channels': 2, 'sample_width': 2, 'frames': 479688, 'duration': 9.9935}


#### 全件取得

In [55]:
def save_acquisition_log(new_rows):

    new_df = pd.DataFrame(new_rows)

    if LOG_CSV.exists():

        old_df = pd.read_csv(
            LOG_CSV
        )

        combined = pd.concat(
            [
                old_df,
                new_df,
            ],
            ignore_index=True,
        )

    else:
        combined = new_df

    combined = (
        combined
        .drop_duplicates(
            subset=["clip_id"],
            keep="last"
        )
        .sort_values("clip_id")
        .reset_index(drop=True)
    )

    combined.to_csv(
        LOG_CSV,
        index=False
    )

    return combined

In [56]:
results_buffer = []

for i, (_, row) in enumerate(
    source_clips.iterrows(),
    start=1,
):

    clip_id = row["clip_id"]

    print(
        f"[{i}/{len(source_clips)}] "
        f"{clip_id}"
    )

    result = download_audioset_clip(
        row
    )

    results_buffer.append(result)

    print(
        " ->",
        result["status"]
    )

    # 10件ごとにDriveへログ保存
    if len(results_buffer) >= 10:

        save_acquisition_log(
            results_buffer
        )

        results_buffer = []

    time.sleep(1)


# 最後の端数
if results_buffer:
    save_acquisition_log(
        results_buffer
    )

print("AudioSet download pass completed.")

[1/465] -7xDzCwyAco_30000
 -> existing
[2/465] -81GPDp9IQY_20000
 -> download_error
[3/465] -DexfVugZDE_0
 -> short_clip
[4/465] -NrpfrEGq5w_40000
 -> existing
[5/465] -NsUQ2A0ClQ_0
 -> existing
[6/465] 01i68fQIo1o_100000
 -> download_error
[7/465] 06rCpqwkcDw_10000
 -> available
[8/465] 0Cd8q2Zg1Mc_25000
 -> short_clip
[9/465] 0IMorrErsv0_50000
 -> available
[10/465] 0NgKGcIkHPM_50000
 -> available
[11/465] 0NvRnSZo_yw_0
 -> available
[12/465] 0SrJ2zPEqLs_30000
 -> available
[13/465] 0VFkSevVAqw_30000
 -> available
[14/465] 0WnknjrATdA_40000
 -> download_error
[15/465] 0kNCB0oGiL4_0
 -> available
[16/465] 0mgIsS0wKro_30000
 -> download_error
[17/465] 0uuCpTLL594_27000
 -> download_error
[18/465] 1-TnkdEPXEw_20000
 -> available
[19/465] 1HQiWO_XJ6M_1000
 -> download_error
[20/465] 1JNYLJtHR5U_30000
 -> available
[21/465] 1XTHCpBJLFQ_40000
 -> available
[22/465] 1YoChLI2bkg_10000
 -> available
[23/465] 1aGAirBjaNg_7000
 -> short_clip
[24/465] 1bOXPrzHVlc_70000
 -> available
[25/465] 1fZ

In [57]:
log = pd.read_csv(LOG_CSV)

print(log["status"].value_counts(dropna=False))

usable = log[
    log["status"].isin(["available", "existing"])
]

print("Usable source clips:", len(usable))
print("Total source clips :", len(source_clips))
print("Availability rate  :", len(usable) / len(source_clips))

status
available         383
download_error     61
short_clip         18
existing            3
Name: count, dtype: int64
Usable source clips: 386
Total source clips : 465
Availability rate  : 0.8301075268817204


In [58]:
usable_clip_ids = set(usable["clip_id"])

usable_events = snore_events[
    snore_events["clip_id"].isin(usable_clip_ids)
]

print("Usable events:", len(usable_events))
print(
    "Usable source clips:",
    usable_events["clip_id"].nunique()
)

print()
print(
    usable_events["official_split"]
    .value_counts()
)

Usable events: 1109
Usable source clips: 386

official_split
train    971
eval     138
Name: count, dtype: int64


#### manifest作成

In [59]:
from pathlib import Path
import pandas as pd
import wave

# ============================================================
# Path
# ============================================================

RAW_ROOT = Path(
    "/content/drive/MyDrive/TRON/AI/dataset/raw/audioset_snoring"
)

META_ROOT = Path(
    "/content/drive/MyDrive/TRON/AI/dataset/metadata/audioset_snoring"
)

SOURCE_CLIPS_CSV = META_ROOT / "snoring_source_clips.csv"
EVENTS_OFFICIAL_CSV = META_ROOT / "snoring_events_official.csv"
ACQUISITION_LOG_CSV = META_ROOT / "acquisition_log.csv"

SOURCE_MANIFEST_CSV = META_ROOT / "source_manifest.csv"
EVENTS_CSV = META_ROOT / "events.csv"


# ============================================================
# Utility
# ============================================================

def get_wav_info(path):
    try:
        with wave.open(str(path), "rb") as wf:
            sr = wf.getframerate()
            channels = wf.getnchannels()
            frames = wf.getnframes()
            sample_width = wf.getsampwidth()

        return {
            "original_sr": sr,
            "original_channels": channels,
            "sample_width_bytes": sample_width,
            "duration_sec": frames / sr if sr > 0 else None,
        }

    except Exception:
        return {
            "original_sr": None,
            "original_channels": None,
            "sample_width_bytes": None,
            "duration_sec": None,
        }


# ============================================================
# Load
# ============================================================

source_clips = pd.read_csv(SOURCE_CLIPS_CSV)
acq = pd.read_csv(ACQUISITION_LOG_CSV)

print("Official source clips:", len(source_clips))
print("Acquisition log rows :", len(acq))


# ============================================================
# Merge acquisition result
# ============================================================

# acquisition_log側から必要な列だけ
acq_cols = [
    "clip_id",
    "status",
    "relative_path",
    "duration_sec",
    "error",
]

acq_small = acq[acq_cols].copy()

source = source_clips.merge(
    acq_small,
    on="clip_id",
    how="left",
    validate="one_to_one",
)


# ============================================================
# Build source manifest
# ============================================================

rows = []

for _, row in source.iterrows():

    clip_id = str(row["clip_id"])
    youtube_id = str(row["youtube_id"])
    acquisition_status = row["status"]

    usable = acquisition_status in {
        "available",
        "existing",
    }

    if usable:
        wav_path = RAW_ROOT / f"{clip_id}.wav"

        if wav_path.exists():
            info = get_wav_info(wav_path)
            relative_path = wav_path.relative_to(RAW_ROOT).as_posix()
            size_bytes = wav_path.stat().st_size
        else:
            # logではusableなのに実ファイルがない場合
            info = {
                "original_sr": None,
                "original_channels": None,
                "sample_width_bytes": None,
                "duration_sec": None,
            }

            relative_path = ""
            size_bytes = None
            usable = False
            acquisition_status = "missing_file"

    else:
        info = {
            "original_sr": None,
            "original_channels": None,
            "sample_width_bytes": None,
            "duration_sec": None,
        }

        relative_path = ""
        size_bytes = None

    availability = (
        "available"
        if usable
        else "unavailable"
    )

    rows.append({

        # ====================================================
        # Common fields
        # ====================================================

        "source_id":
            f"audioset:{clip_id}",

        "source_dataset":
            "AudioSet Strong",

        # Strong annotation datasetを示す。
        # 後で必要なら取得日等を追加してもよい。
        "dataset_version":
            "AudioSet Strong",

        "source_recording_id":
            clip_id,

        "subject_id":
            pd.NA,

        "session_id":
            pd.NA,

        "original_label":
            "Snoring",

        "class_id":
            1,

        "class_name":
            "SNORE",

        "usage_group":
            "target",

        "relative_path":
            relative_path,

        "original_sr":
            info["original_sr"],

        "original_channels":
            info["original_channels"],

        "sample_width_bytes":
            info["sample_width_bytes"],

        "duration_sec":
            info["duration_sec"],

        "size_bytes":
            size_bytes,

        "quality":
            pd.NA,

        # AudioSet Strong annotationそのもの
        "annotation_license":
            "CC BY 4.0",

        # 元音声はYouTube由来なので別管理
        "audio_source":
            "YouTube",

        "audio_license":
            "source-dependent",

        "source_url":
            f"https://www.youtube.com/watch?v={youtube_id}",

        # 同じ10秒clip由来の全eventを同じsplitへ
        "provenance_group_id":
            f"audioset:{clip_id}",

        "availability":
            availability,

        "split":
            "UNASSIGNED",

        "notes":
            "",


        # ====================================================
        # AudioSet-specific fields
        # ====================================================

        "youtube_id":
            youtube_id,

        "audioset_clip_id":
            clip_id,

        "audioset_clip_start_sec":
            float(row["clip_start_sec"]),

        "official_split":
            row["official_split"],

        # download処理上の生の結果
        "acquisition_status":
            acquisition_status,

        "acquisition_error":
            row["error"] if pd.notna(row["error"]) else "",
    })


source_manifest = pd.DataFrame(rows)

source_manifest["class_id"] = (
    source_manifest["class_id"]
    .astype("Int64")
)

source_manifest.to_csv(
    SOURCE_MANIFEST_CSV,
    index=False,
)

print()
print("Saved:")
print(SOURCE_MANIFEST_CSV)

Official source clips: 465
Acquisition log rows : 465

Saved:
/content/drive/MyDrive/TRON/AI/dataset/metadata/audioset_snoring/source_manifest.csv


In [60]:
print("=== Basic check ===")

print("Rows            :", len(source_manifest))
print(
    "Available       :",
    (source_manifest["availability"] == "available").sum()
)
print(
    "Unavailable     :",
    (source_manifest["availability"] == "unavailable").sum()
)

print()
print("=== Acquisition status ===")

print(
    source_manifest["acquisition_status"]
    .value_counts(dropna=False)
)

print()
print("=== Audio format (usable only) ===")

usable_sources = source_manifest[
    source_manifest["availability"] == "available"
]

print(
    usable_sources[
        [
            "original_sr",
            "original_channels",
            "sample_width_bytes",
        ]
    ]
    .value_counts()
)

print()
print("=== Duration ===")

print(
    usable_sources["duration_sec"]
    .describe()
)

print()
print("=== Official split ===")

print(
    source_manifest["official_split"]
    .value_counts()
)


# Assertions

assert len(source_manifest) == 465
assert source_manifest["source_id"].is_unique

assert (
    source_manifest["availability"] == "available"
).sum() == 386

assert (
    source_manifest["class_id"] == 1
).all()

assert (
    source_manifest["class_name"] == "SNORE"
).all()

assert (
    source_manifest["usage_group"] == "target"
).all()

print()
print("AudioSet source manifest check: OK")

=== Basic check ===
Rows            : 465
Available       : 386
Unavailable     : 79

=== Acquisition status ===
acquisition_status
available         383
download_error     61
short_clip         18
existing            3
Name: count, dtype: int64

=== Audio format (usable only) ===
original_sr  original_channels  sample_width_bytes
48000.0      2.0                2.0                   223
44100.0      2.0                2.0                   152
             1.0                2.0                    11
Name: count, dtype: int64

=== Duration ===
count    386.000000
mean       9.999274
std        0.002990
min        9.953923
25%       10.000000
50%       10.000000
75%       10.000000
max       10.000023
Name: duration_sec, dtype: float64

=== Official split ===
official_split
train    410
eval      55
Name: count, dtype: int64

AudioSet source manifest check: OK


In [61]:
events = pd.read_csv(
    EVENTS_OFFICIAL_CSV
)

# 時刻型を念のため保証
events["event_start"] = pd.to_numeric(
    events["event_start"],
    errors="coerce",
)

events["event_end"] = pd.to_numeric(
    events["event_end"],
    errors="coerce",
)

events["event_duration"] = (
    events["event_end"]
    - events["event_start"]
)


# ============================================================
# source availabilityを結合
# ============================================================

source_status = source_manifest[
    [
        "audioset_clip_id",
        "source_id",
        "availability",
        "acquisition_status",
        "provenance_group_id",
    ]
].copy()

source_status = source_status.rename(
    columns={
        "audioset_clip_id": "clip_id"
    }
)

events = events.merge(
    source_status,
    on="clip_id",
    how="left",
    validate="many_to_one",
)


# ============================================================
# Event ID
# ============================================================

# 同一clip内でevent番号を振る
events = events.sort_values(
    [
        "clip_id",
        "event_start",
        "event_end",
    ]
).reset_index(drop=True)

events["event_index"] = (
    events
    .groupby("clip_id")
    .cumcount()
)

events["event_id"] = events.apply(
    lambda row:
        f"audioset:{row['clip_id']}:event:{int(row['event_index']):03d}",
    axis=1,
)


# ============================================================
# TRON class information
# ============================================================

events["class_id"] = 1
events["class_name"] = "SNORE"

# AI-03でclip化するまではanchor未確定
events["event_anchor_sec"] = pd.NA

# AI-05で分割するのでまだ未割当
events["split"] = "UNASSIGNED"


# ============================================================
# Column order
# ============================================================

events = events[
    [
        "event_id",
        "source_id",
        "clip_id",

        "youtube_id",
        "clip_start_sec",

        "official_split",

        "event_index",
        "event_start",
        "event_end",
        "event_duration",

        "mid",

        "class_id",
        "class_name",

        "event_anchor_sec",

        "provenance_group_id",

        "availability",
        "acquisition_status",

        "split",
    ]
]


events["class_id"] = (
    events["class_id"]
    .astype("Int64")
)

events.to_csv(
    EVENTS_CSV,
    index=False,
)

print("Saved:")
print(EVENTS_CSV)

Saved:
/content/drive/MyDrive/TRON/AI/dataset/metadata/audioset_snoring/events.csv


In [62]:
print("=== Event check ===")

print("Total events:", len(events))

print(
    "Unique source clips:",
    events["clip_id"].nunique()
)

print()
print("=== Availability ===")

print(
    events["availability"]
    .value_counts(dropna=False)
)

print()
print("=== Usable events ===")

usable_events = events[
    events["availability"] == "available"
]

print(
    "Usable events:",
    len(usable_events)
)

print(
    "Usable source clips:",
    usable_events["clip_id"].nunique()
)

print()
print("=== Usable official split ===")

print(
    usable_events["official_split"]
    .value_counts()
)

print()
print("=== Event duration ===")

print(
    usable_events["event_duration"]
    .describe()
)


# ============================================================
# Assertions
# ============================================================

assert len(events) == 1316

assert events["event_id"].is_unique

assert events["clip_id"].nunique() == 465

assert len(usable_events) == 1109

assert usable_events["clip_id"].nunique() == 386

assert (
    usable_events["official_split"]
    .value_counts()
    .to_dict()
    == {
        "train": 971,
        "eval": 138,
    }
)

assert events["event_start"].notna().all()
assert events["event_end"].notna().all()

assert (
    events["event_duration"] > 0
).all()

print()
print("AudioSet events check: OK")

=== Event check ===
Total events: 1316
Unique source clips: 465

=== Availability ===
availability
available      1109
unavailable     207
Name: count, dtype: int64

=== Usable events ===
Usable events: 1109
Usable source clips: 386

=== Usable official split ===
official_split
train    971
eval     138
Name: count, dtype: int64

=== Event duration ===
count    1109.000000
mean        1.238055
std         0.768858
min         0.059000
25%         0.742000
50%         1.189000
75%         1.556000
max        10.000000
Name: event_duration, dtype: float64

AudioSet events check: OK
